# Zadanie 2.3: warunki brzegowe i kształt guza na starcie

Sprawdzamy, kiedy brzeg obszaru i kształt startowy guza zmieniają masę $M(t) = \int u\,dx$ (issue #11).

- **Warunek brzegowy:** Neumann ($\partial_x u = 0$, brak przepływu, model bazowy) albo Dirichlet ($u = 0$, brzeg pochłania komórki,
  nowa funkcja `laplacian_dirichlet` i argument `bc` w `simulate`).
- **Wielkość obszaru:** długość $L$ od 16 do 80 przy guzie o promieniu 6 w środku.
- **Kształt guza:** plateau, gaussian, dwa ogniska i guz przy brzegu obszaru, wszystkie o tej samej masie początkowej.

Harmonogram jak w `zad2_02_scenariusze`: 10 frakcji co 0,4 od $t = 1$, $\beta = 1{,}5$, solver B (`cn`), $\Delta x = 0{,}1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import NO_TREATMENT, Params, Schedule, beam, grid, initial_condition
from offtarget.observables import min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

DX = 0.1
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
BCS = {"neumann": "Neumann", "dirichlet": "Dirichlet"}


def run(params, sched, x, u0, bc):
    return tumor_mass(simulate(params, sched, x, t, u0=u0, method="cn", dose="pulse", bc=bc), x)


def summary(M):
    m_min, t_min = min_mass(t, M)
    return {"M_min / M(0)": m_min / M[0], "czas do odrostu": time_to_regrowth(t, M), "M(10) / M(0)": M[-1] / M[0]}

## Wielkość obszaru i warunek brzegowy

Guz: plateau o promieniu 6 w środku obszaru $[-L/2, L/2]$. Dwa plany: bez leczenia i pole przesunięte ($s = 6$, $r = 8$),
przy którym lewy brzeg guza zostaje poza polem i odrasta.

In [ ]:
lengths = [16.0, 24.0, 40.0, 80.0]
plans = {"bez leczenia": (Params(), NO_TREATMENT), "pole przesunięte": (Params(beta=1.5, s=6.0, r=8.0), schedule)}

size_runs, rows = {}, []
for L in lengths:
    x, _ = grid(L, int(round(L / DX)))
    u0 = initial_condition(x)
    for plan, (p, sched) in plans.items():
        for bc in BCS:
            M = run(p, sched, x, u0, bc)
            size_runs[(L, plan, bc)] = M
            rows.append({"L": L, "plan": plan, "brzeg": BCS[bc], **summary(M), "M(10) / L": M[-1] / L})
size_table = pd.DataFrame(rows)
size_table.to_csv(TABLES / "zad2_02_warunki_obszar.csv", index=False)
size_table.style.format(precision=3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, plan in zip(axes, plans):
    for L, c in zip(lengths, plt.cm.viridis(np.linspace(0, 0.85, len(lengths)))):
        for bc, ls in zip(BCS, ["-", "--"]):
            M = size_runs[(L, plan, bc)]
            ax.plot(t, M / M[0], color=c, ls=ls, lw=2 if ls == "-" else 1.5)
    ax.set(xlabel="czas t", title=plan)
for L, c in zip(lengths, plt.cm.viridis(np.linspace(0, 0.85, len(lengths)))):
    axes[0].plot([], [], color=c, lw=2, label=f"L = {L:g}")
axes[0].plot([], [], "k-", label="Neumann")
axes[0].plot([], [], "k--", label="Dirichlet")
axes[0].set_ylabel("M(t) / M(0)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_02_warunki_obszar.png", dpi=150)

## Kształt guza na starcie

Cztery kształty o tej samej masie co plateau o promieniu 6 (obszar $L = 80$):

- **plateau**: $u_0$ z `initial_condition`, promień 6;
- **gaussian**: $u_0 = e^{-x^2 / (2\sigma^2)}$, $\sigma$ dobrane do tej samej masy, więc ma długie ogony;
- **dwa ogniska**: dwa plateau o promieniu 3 w $x = \pm 8$;
- **przy brzegu**: plateau o promieniu 6 ze środkiem 7 od prawego brzegu.

Plany są ustawione względem środka guza $c$: pole pokrywa guz ($s = c$, $r = 8$) oraz pole przesunięte o 6 ($s = c + 6$, $r = 8$).
Dla guza przy brzegu liczymy też przesunięcie w drugą stronę ($s = c - 6$), żeby nienapromieniony brzeg guza dotykał brzegu obszaru.

In [ ]:
L = 80.0
x, _ = grid(L, int(round(L / DX)))
plateau = initial_condition(x)
M0 = tumor_mass(plateau, x)
sigma = M0 / np.sqrt(2 * np.pi)
edge_center = L / 2 - 7.0
shapes = {
    "plateau": (plateau, 0.0),
    "gaussian": (np.exp(-(x**2) / (2 * sigma**2)), 0.0),
    "dwa ogniska": (initial_condition(x, center=-8.0, radius=3.0) + initial_condition(x, center=8.0, radius=3.0), 0.0),
    "przy brzegu": (initial_condition(x, center=edge_center), edge_center),
}
masses = {name: tumor_mass(u0, x) for name, (u0, _) in shapes.items()}
print({name: round(m / M0, 3) for name, m in masses.items()}, f"sigma = {sigma:.2f}")

In [ ]:
shape_runs, rows = {}, []
for name, (u0, c) in shapes.items():
    shape_plans = {"pole pokrywa guz": Params(beta=1.5, s=c, r=8.0), "pole przesunięte o +6": Params(beta=1.5, s=c + 6.0, r=8.0)}
    if name == "przy brzegu":
        shape_plans["pole przesunięte o -6"] = Params(beta=1.5, s=c - 6.0, r=8.0)
    for plan, p in shape_plans.items():
        for bc in BCS:
            M = run(p, schedule, x, u0, bc)
            shape_runs[(name, plan, bc)] = M
            rows.append({"kształt": name, "plan": plan, "brzeg": BCS[bc], **summary(M)})
shape_table = pd.DataFrame(rows)
shape_table.to_csv(TABLES / "zad2_02_warunki_ksztalt.csv", index=False)
shape_table.style.format(precision=3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for (name, (u0, c)), color in zip(shapes.items(), plt.cm.tab10.colors):
    axes[0].plot(x, u0, color=color, lw=2, label=name)
axes[0].set(xlabel="x", ylabel="u(x, 0)", title="Kształty startowe (ta sama masa)")
axes[0].legend(fontsize=8)
for ax, plan in zip(axes[1:], ["pole pokrywa guz", "pole przesunięte o +6"]):
    for name, color in zip(shapes, plt.cm.tab10.colors):
        M = shape_runs[(name, plan, "neumann")]
        ax.plot(t, M / M[0], color=color, lw=2, label=name)
    ax.set(xlabel="czas t", ylabel="M(t) / M(0)", title=f"{plan} (Neumann)", ylim=(0, None))
fig.tight_layout()
fig.savefig(FIGURES / "zad2_02_warunki_ksztalt.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for plan, color in zip(["pole pokrywa guz", "pole przesunięte o +6", "pole przesunięte o -6"], plt.cm.tab10.colors):
    for bc, ls in zip(BCS, ["-", "--"]):
        M = shape_runs[("przy brzegu", plan, bc)]
        ax.plot(t, M / M[0], color=color, ls=ls, lw=2, label=f"{plan}, {BCS[bc]}")
ax.set(xlabel="czas t", ylabel="M(t) / M(0)", title="Guz 7 od prawego brzegu obszaru", ylim=(0, None))
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_02_warunki_brzeg.png", dpi=150)

## Wnioski

- **Brzeg nie ma znaczenia, gdy guz jest daleko od niego.** Przy $L = 80$ Neumann i Dirichlet dają identyczne $M(t)$
  (bez leczenia $M(10)/M(0) = 3{,}81$, pole przesunięte $2{,}65$). Front dociera na odległość około 20 od środka,
  więc obszar $[-40, 40]$ jest dla tych scenariuszy praktycznie nieskończony.
- **W małym obszarze brzeg ogranicza wzrost.** Przy Neumannie masa nasyca się na $L \cdot K$ (dla $L = 16$ cały obszar jest zapełniony,
  $M(10)/L = 1$), a wzrost bez leczenia spada z 3,81 do 1,36. Dirichlet dodatkowo zabiera komórki przez brzeg:
  dla $L = 16$ bez leczenia $M(10)/M(0)$ spada z 1,36 do 1,15, a po polu przesuniętym guz nie wraca do $M(0)$ do $t = 10$.
  Od $L = 40$ minimum masy i czas do odrostu nie zależą już od warunku brzegowego.
- **Kształt startowy zmienia wynik bardziej niż brzeg.** Przy tej samej masie i polu pokrywającym guz ($r = 8$)
  plateau spada do 0,16 $M(0)$ i odrasta po $t = 5{,}7$, gaussian tylko do 0,38 (odrost po 4,15), a dwa ogniska do 0,78 (odrost po 2,8),
  bo ogony gaussiana i zewnętrzne połowy ognisk leżą poza polem. Pole pokryło część guza, ale nie jego zasięg.
- **Guz przy brzegu rośnie wolniej** (tylko jeden front): pole pokrywające daje $M(10)/M(0) = 1{,}75$ zamiast 3,38 dla plateau w środku.
  Warunek brzegowy ma znaczenie tylko wtedy, gdy przy brzegu zostają żywe komórki: przy polu przesuniętym od brzegu ($s = c - 6$)
  Dirichlet obniża minimum z 0,36 do 0,24 $M(0)$ i opóźnia odrost z 9,0 do 9,4. Przy polu przesuniętym w stronę brzegu różnica jest pomijalna.
- **Wniosek dla dalszych zadań:** model bazowy (Neumann, $L = 80$, guz w środku) nie jest wrażliwy na warunek brzegowy,
  za to wynik leczenia mocno zależy od tego, jaką część zasięgu guza obejmuje pole.